# Numerical Comparison of Restart Methods of the Arnoldi Method

## Imports

In [ ]:
using LinearAlgebra
using JacobiDavidson
using LinearMaps
using MatrixDepot
ENV["GKSwstype"] = "nul"
using Plots
using Plots.Measures
using ProgressMeter
using LaTeXStrings

include("../src/Orthogonalization.jl")
include("../src/Arnoldi.jl")
include("../src/ImplicitRestart.jl")
include("../src/Eigenpairs.jl")
include("../src/BadRestart.jl")

## IRAM vs Jacobi-Davidson

We use the large sparse matrix "rajat12" from MatrixDepot to compare the accuracy of IRAM and Jacobi-Davidson as a function of the number of iterations and subspace dimension.

In [ ]:
A = matrixdepot(r"rajat12") + 10 * I
n = size(A, 1)
num_eig = 4

display(A)

In [ ]:
evals = eigvals(Matrix(A))

println(minimum(real(evals)))
println(maximum(real(evals)))
println(minimum(imag(evals)))
println(maximum(imag(evals)))

p = scatter(real(evals), imag(evals), color=:blue,
     title = "Eigenvalues of rajat12 matrix", xlabel="Real Part", ylabel="Imaginary Part",
     xlim=[-100, 1100], ylim = [-1e-2, 1e-2], legend=:top,
     markerstrokewidth=0, markersize=3, label="")

savefig(p, "../fig/IRAM vs Jacobi-Davidson/evals.png")

In [ ]:
maximum_iterations = 50
maximum_subspace = 30
mean_iterations = 5

iter_grid = Int.(range(1, maximum_iterations, maximum_iterations))
subspace_grid = Int.(range(6, maximum_subspace, 13))

residuals_iram = zeros(length(iter_grid), length(subspace_grid))
residuals_jd = zeros(length(iter_grid), length(subspace_grid))

it = 1

@showprogress for s in subspace_grid
    for _ in 1:mean_iterations
        _, _, residual_iram = Eigenpairs.eigenpairs_iram(A, n, num_eig=num_eig, max_iter=maximum_iterations, subspace_dim=s, restart_dim=max(floor(Int, s/2), num_eig), tol=0.0);
        _, _, residual_jd = Eigenpairs.eigenpairs_jd(A, n, num_eig=num_eig, max_iter=maximum_iterations, subspace_dim=s, restart_dim=max(floor(Int, s/2), num_eig), tol=0.0);
        residuals_iram[:, it] += residual_iram
        residuals_jd[:, it] += residual_jd
    end
    it += 1
end

residuals_iram ./= mean_iterations
residuals_jd ./= mean_iterations;

In [ ]:
println("Range of IRAM residuals: ", extrema(log10.(residuals_iram)))
println("Range of Jacobi-Davidson residuals: ", extrema(log10.(residuals_jd)))

In [ ]:
p = surface(iter_grid, subspace_grid, log10.(residuals_iram'), 
        xlabel="Restart iterations              ", 
        ylabel="Subspace dimensions", 
        zlabel="Residual",
        title="Log Residuals of IRAM",
        colorbar = true,
        alpha = 1,
        zlim=(-18, 2),
        camera=(60, 30),
        ratio=:equal,
        right_margin = 12 * Measures.mm)

savefig(p, "../fig/IRAM vs Jacobi-Davidson/Residuals_IRAM.png")

In [ ]:
p = surface(iter_grid, subspace_grid, log10.(residuals_jd'), 
        xlabel="Restart iterations              ", 
        ylabel="Subspace dimensions", 
        zlabel="Residual",
        title="Log Residuals of Jacobi-Davidson",
        colorbar = true,
        #colormap = :viridis, # :magma, :inferno, :turbo are also great
        alpha = 1,
        zlim=(-18, 2),
        camera=(60, 30),
        ratio=:equal,
        right_margin = 12 * Measures.mm)

savefig(p, "../fig/IRAM vs Jacobi-Davidson/Residuals_Jacobi-Davidson.png")

In [ ]:
p = plot(iter_grid, residuals_iram[:, end], color=:blue,
     title = L"Residual $\max_i \, |\!|Av_i - \lambda_i v_i|\!|_1$", xlabel="Restart iterations", ylabel="Residual",
     yaxis=:log,
     xlim=[1, maximum_iterations], ylim = [1e-18, 1e3], legend=:topright,
     label = "IRAM (dimension 50)")

plot!(p, iter_grid, residuals_jd[:, end], color=:green,
     label = "Jacobi-Davidson (dimension 50)")

plot!(p, iter_grid, residuals_iram[:, 1], color=:red,
     label = "IRAM (dimension 6)")

plot!(p, iter_grid, residuals_jd[:, 1], color=:orange,
     label = "Jacobi-Davidson (dimension 6)")

savefig(p, "../fig/IRAM vs Jacobi-Davidson/Residuals_subspace.png")

In [ ]:
p = plot(subspace_grid, residuals_iram[end, :], color=:blue,
     title = L"Residual $\max_i \, |\!|Av_i - \lambda_i v_i|\!|_1$", xlabel="Subspace dimensions", ylabel="Residual",
     yaxis=:log,
     xlim=[6, maximum_subspace], ylim = [1e-18, 1e2], legend=:top,
     label = "IRAM")

plot!(p, subspace_grid, residuals_jd[end, :], color=:green,
     label = "Jacobi-Davidson")

savefig(p, "../fig/IRAM vs Jacobi-Davidson/Residuals_iteration.png")